In [5]:
# ===== Step 0: 导入库 =====
import os
import librosa
import numpy as np
import pandas as pd

# ===== Step 1: 设置路径 =====
audio_folder = "wav_files"   # 你的 wav 文件所在目录
metadata_file = "metadata_2.xlsx"   # 含 V/A/S 标签的 Excel
output_csv = "features_02.csv"       # 输出 CSV 文件

# ===== Step 2: 读取 metadata =====
metadata = pd.read_excel(metadata_file)
print("Metadata preview:")
print(metadata.head())

# ===== Step 3: 定义特征提取函数 =====
def extract_features(file_path, sr=22050, n_mfcc=13):
    """
    提取单条音频特征:
    - MFCC (平均 + 标准差)
    - Chroma (平均)
    - Zero Crossing Rate (平均)
    - RMS 能量 (平均)
    - Spectral Centroid (平均)
    """
    y, sr = librosa.load(file_path, sr=sr, mono=True)
    
    # ---- MFCC ----
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfccs.mean(axis=1)
    mfcc_std = mfccs.std(axis=1)
    
    # ---- Chroma ----
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = chroma.mean(axis=1)
    
    # ---- Zero Crossing Rate ----
    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = zcr.mean()
    
    # ---- RMS 能量 ----
    rms = librosa.feature.rms(y=y)
    rms_mean = rms.mean()
    
    # ---- Spectral Centroid ----
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_cent_mean = spec_cent.mean()
    
    # ---- 整合特征 ----
    feature_vector = np.concatenate([
        mfcc_mean, mfcc_std, chroma_mean, 
        [zcr_mean, rms_mean, spec_cent_mean]
    ])
    
    return feature_vector

# ===== Step 4: 提取所有音频特征 =====
feature_list = []

for idx, row in metadata.iterrows():
    # 自动加上 .wav 后缀
    fname = str(row['FileName']) + ".wav"
    path = os.path.join(audio_folder, fname)
    
    if os.path.exists(path):
        features = extract_features(path)
        feature_list.append(features)
    else:
        print(f"Warning: {fname} not found! Skipping.")

# ===== Step 5: 构建 DataFrame =====
feature_array = np.array(feature_list)

# 特征列命名
mfcc_cols = [f"mfcc{i+1}_mean" for i in range(13)] + [f"mfcc{i+1}_std" for i in range(13)]
chroma_cols = [f"chroma{i+1}" for i in range(12)]
other_cols = ["zcr", "rms", "spectral_centroid"]
all_feature_cols = mfcc_cols + chroma_cols + other_cols

features_df = pd.DataFrame(feature_array, columns=all_feature_cols)

# 只保留有对应音频的 metadata 行
metadata_with_audio = metadata.iloc[:len(features_df)].reset_index(drop=True)

# 合并 V/A/S 与特征
full_df = pd.concat([metadata_with_audio, features_df], axis=1)

# ===== Step 6: 保存到 CSV =====
full_df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"Feature extraction completed! Saved to {output_csv}")


Metadata preview:
   Number                           FileName  Valence  Arousal  \
0       1  Aster_alarmrespondingtohuman_1524     0.86      2.0   
1       2          Aster_alertoncurtain_1906    -0.50      3.8   
2       3         Aster_aloneresponding_1458     0.23      1.0   
3       4            Aster_alonetalking_1335     0.00      1.8   
4       5     Aster_angryathomeoccupied_1904    -3.50      4.0   

   SocialEngagement  Time           Behavior Gender  Age  
0               2.5  1524  respondingtohuman      M  1.5  
1              -0.5  1906              alert      M  1.5  
2               1.5  1458         responding      M  1.5  
3              -2.0  1335           selftalk      M  1.5  
4              -1.5  1904              angry      M  1.5  
Feature extraction completed! Saved to features_02.csv
